In [3]:
import numpy as np
import pandas as pd

np.random.seed(42)
N = 6000
data = []

for _ in range(N):
    temp_drift = np.random.uniform(0, 0.3)
    excursion_time = np.random.uniform(0, 90)
    temp_variance = np.random.uniform(0.1, 8)
    door_rate = np.random.uniform(0, 12)
    handling_index = 0.6*door_rate + np.random.uniform(0,4)
    delay_risk = np.random.uniform(0, 1)
    heat_stress = np.random.uniform(10, 45)
    thermal_buffer = np.random.uniform(0.2, 2.5)

    score = (
        0.6*temp_drift +
        0.025*excursion_time +
        0.12*temp_variance +
        0.08*door_rate +
        0.18*delay_risk +
        0.04*heat_stress -
        0.35*thermal_buffer
    )

    # probabilistic decision (adds class diversity)
    prob = 1 / (1 + np.exp(-score))
    spoilage = 1 if np.random.rand() < prob else 0

    data.append([
        temp_drift, excursion_time, temp_variance,
        door_rate, handling_index,
        delay_risk, heat_stress, thermal_buffer,
        spoilage
    ])

df = pd.DataFrame(data, columns=[
    "temp_drift","excursion_time","temp_variance",
    "door_rate","handling_index",
    "delay_risk","heat_stress","thermal_buffer",
    "spoilage_label"
])

df.to_csv("synthetic_transport.csv", index=False)
print("Synthetic dataset regenerated")


Synthetic dataset regenerated


In [5]:

import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import joblib

df = pd.read_csv("synthetic_transport.csv")

X = df.drop(columns=["spoilage_label"])
y = df["spoilage_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:,1]))
joblib.dump(model, "transport_risk.pkl")


AUC: 0.7157535978316932


['transport_risk.pkl']

uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload


In [1]:
# convert_to_header.py

with open("student_model.tflite", "rb") as f:
    data = f.read()

with open("model.h", "w") as f:
    f.write("unsigned char model_tflite[] = {\n")

    for i, b in enumerate(data):
        if i % 12 == 0:
            f.write("\n ")
        f.write(f"0x{b:02x}, ")

    f.write("\n};\n")
    f.write(f"unsigned int model_tflite_len = {len(data)};\n")

print("model.h generated!")

model.h generated!
